[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ELTE-DSED/Intro-Data-Security/blob/main/modules/module_06_synthetic_data/Lab_6_Tabular_Synthetic_Data_Generation.ipynb)

# **Lab 6: Tabular Synthetic Data Generation (VAE & CTGAN)**

**Master's Course:** Introduction to Data Security Practicum  
**Institution:** Eötvös Loránd University (ELTE), Faculty of Informatics (IK)  
**Module 6:** Privacy-Preserving Synthetic Data


## **Learning Objectives**

By the end of this lab, you will be able to:

- **Understand Synthetic Data Threat Models:** Analyze how sharing synthetic tabular data protects individual record privacy while preserving downstream analytical utility.
- **Implement Deep Generative Models:** Build and train Conditional Variational Autoencoders (CVAE) and Conditional Generative Adversarial Networks (CGAN) for tabular feature synthesis.
- **Evaluate Statistical Fidelity:** Quantify distribution matching using feature mean absolute error, correlation matrix similarity, and covariance alignment.
- **Measure Downstream Utility (TSTR):** Implement the Train-on-Synthetic, Test-on-Real (TSTR) paradigm to verify downstream classification performance against real-world test sets.
- **Audit Privacy & MIA Resistance:** Quantify Nearest-Neighbor Distance Ratios (NNDR) and empirical memorization to ensure synthetic records do not leak raw training samples.
- **Implement Distance Filtering:** Deploy threshold-based sample rejection guards to sanitize synthetic batches and mitigate Membership Inference vulnerabilities.


## **Table of Contents**

1. [Threat Model & Privacy-Preserving Synthetic Data](#threat-model)
2. [Generative Modeling Foundations (CVAE vs. CGAN/CTGAN)](#theory)
3. [Setup & Data Preparation (Iris Tabular Benchmark)](#setup)
4. [Privacy Safeguards & Distance Filtering](#privacy-guard)
5. [Conditional Variational Autoencoder (CVAE)](#cvae)
6. [Conditional Generative Adversarial Network (CGAN)](#cgan)
7. [Fidelity & Utility Evaluation (TSTR Paradigm)](#evaluation)
8. [Membership Inference Attack (MIA) Resistance](#mia-defense)
9. [Dimensionality Reduction & Distribution Verification (PCA)](#pca)
10. [Hands-on Exploratory Exercises](#exercises)
11. [Conclusion & Next Steps](#conclusion)


## **1. Threat Model & Privacy-Preserving Synthetic Data** <a name="threat-model"></a>

When organizations share microdata (clinical trials, financial transactions, user telemetry), anonymization techniques such as $k$-anonymity and pseudonymization frequently fail against linkage attacks.

**Synthetic Data Generation** replaces real records with artificially generated samples that mirror the joint probability distribution $\mathcal{P}(\mathbf{X}, y)$ of the underlying population without copying individual entities.

### **Threat & Defense Model**

- **Attacker Goal:** Re-identify individuals, confirm training set membership (MIA), or reconstruct sensitive attributes from the shared dataset.
- **Defender Goal:** Produce a synthetic dataset $\mathcal{D}_{syn}$ that maximizes:
  1. **Fidelity:** Statistical moments and column correlations match $\mathcal{D}_{real}$.
  2. **Utility:** Classifiers trained on $\mathcal{D}_{syn}$ achieve comparable accuracy on $\mathcal{D}_{real}$ (TSTR).
  3. **Privacy:** Zero 1-to-1 record memorization; synthetic samples lie at safe Euclidean/Hamming distances from real training points.


## **2. Generative Modeling Foundations (CVAE vs. CGAN/CTGAN)** <a name="theory"></a>

Tabular data presents distinct challenges compared to images: mixed continuous and discrete data types, non-Gaussian multimodal distributions, and extreme class imbalances.

In this lab, we build two deep generative architectures:

1. **Conditional Variational Autoencoder (CVAE):**
   - Encodes input features $\mathbf{x}$ and condition $y$ into a continuous latent space parameterized by mean $\boldsymbol{\mu}$ and log-variance $\log \boldsymbol{\sigma}^2$.
   - Optimized via the Evidence Lower Bound (ELBO):
     $$\mathcal{L}_{CVAE} = \mathbb{E}_{q_\phi(\mathbf{z}|\mathbf{x}, y)}[\log p_\theta(\mathbf{x}|\mathbf{z}, y)] - \beta D_{\text{KL}}(q_\phi(\mathbf{z}|\mathbf{x}, y) \parallel p(\mathbf{z}))$$
   - The $\beta$ hyperparameter controls the disentanglement-reconstruction trade-off.

2. **Conditional Generative Adversarial Network (CGAN / CTGAN principles):**
   - Generator $G(\mathbf{z}, y)$ learns to map Gaussian noise $\mathbf{z} \sim \mathcal{N}(0, \mathbf{I})$ conditioned on label $y$ to realistic tabular feature vectors.
   - Discriminator $D(\mathbf{x}, y)$ acts as an adversary distinguishing real tabular rows from generated synthetic rows.
   - Minimax objective:
     $$\min_G \max_D \mathbb{E}_{\mathbf{x} \sim p_{data}}[\log D(\mathbf{x}, y)] + \mathbb{E}_{\mathbf{z} \sim p_z}[\log(1 - D(G(\mathbf{z}, y), y))]$$


## **3. Setup & Data Preparation (Iris Tabular Benchmark)** <a name="setup"></a>

We configure the environment and load the canonical **Fisher's Iris** dataset ($n=150$, $d=4$ continuous features across 3 botanical species).

Standardizing continuous columns ensures stable gradient propagation during GAN and VAE optimization.


In [ ]:
# Install required dependencies for Google Colab
%pip install -q scikit-learn seaborn matplotlib pandas torch


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances
from sklearn.neighbors import KNeighborsClassifier
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
iris = load_iris()
X_real = iris.data
y_real = iris.target

# Standardize
scaler = StandardScaler()
X_real_scaled = scaler.fit_transform(X_real)

# Create PyTorch dataset including labels
train_loader = DataLoader(
    TensorDataset(torch.FloatTensor(X_real_scaled), torch.LongTensor(y_real)), 
    batch_size=32, 
    shuffle=True
)

print(f"Real data shape: {X_real_scaled.shape}")
print(f"Features: {iris.feature_names}")
print(f"Classes: {iris.target_names}")

## **4. Privacy Safeguards & Distance Filtering** <a name="privacy-guard"></a>

A common failure mode in generative synthesis is **empirical memorization**: generative models with high capacity can accidentally reproduce exact training records verbatim.

To establish a defense-in-depth privacy safeguard, we implement `sanitize_synthetic_data`. It calculates pairwise Euclidean distances between candidate synthetic samples and all real training records in normalized feature space, discarding any synthetic record that falls within a minimum proximity radius $r_{min} = 0.30$.


In [ ]:
def sanitize_synthetic_data(synthetic_data_scaled, real_data_scaled, threshold=0.3):
    """
    Privacy Guard: Rejects synthetic samples that are within 'threshold' Euclidean distance 
    of any real training record. Helps pass Membership Inference Analysis.
    """
    dists = pairwise_distances(synthetic_data_scaled, real_data_scaled, metric='euclidean')
    min_dists = dists.min(axis=1)
    safe_indices = np.where(min_dists > threshold)[0]
    return safe_indices

## **5. Conditional Variational Autoencoder (CVAE)** <a name="cvae"></a>

We construct a CVAE with one-hot encoded condition vectors concatenated to both encoder and decoder inputs. The reparameterization trick allows gradients to flow back through stochastic latent samples $\mathbf{z} = \boldsymbol{\mu} + \boldsymbol{\sigma} \odot \boldsymbol{\epsilon}$.


In [ ]:
class CVAE(nn.Module):
    def __init__(self, input_dim=4, latent_dim=2, n_classes=3):
        super(CVAE, self).__init__()
        self.latent_dim = latent_dim
        self.label_emb = nn.Embedding(n_classes, n_classes)
        
        self.encoder = nn.Sequential(
            nn.Linear(input_dim + n_classes, 16),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(16, 8),
            nn.ReLU()
        )
        self.fc_mu = nn.Linear(8, latent_dim)
        self.fc_logvar = nn.Linear(8, latent_dim)
        
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim + n_classes, 8),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, input_dim)
        )
    
    def encode(self, x, labels):
        c = self.label_emb(labels)
        h = self.encoder(torch.cat([x, c], dim=1))
        return self.fc_mu(h), self.fc_logvar(h)
    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def decode(self, z, labels):
        c = self.label_emb(labels)
        return self.decoder(torch.cat([z, c], dim=1))
    
    def forward(self, x, labels):
        mu, logvar = self.encode(x, labels)
        z = self.reparameterize(mu, logvar)
        return self.decode(z, labels), mu, logvar, z
    
    def generate(self, n_samples, labels):
        with torch.no_grad():
            z = torch.randn(n_samples, self.latent_dim).to(device)
            samples = self.decode(z, labels).cpu().numpy()
        return samples


In [ ]:
def vae_loss(recon, x, mu, logvar, beta=15.0):
    recon_loss = nn.functional.mse_loss(recon, x, reduction='sum')
    kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return (recon_loss + beta * kld) / x.size(0)

def train_vae(model, train_loader, epochs=100, lr=1e-3):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    model.train()
    for epoch in range(epochs):
        epoch_loss = 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            recon, mu, logvar, _ = model(x, y)
            loss = vae_loss(recon, x, mu, logvar)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        
        if (epoch + 1) % 20 == 0:
            print(f'Epoch {epoch+1}/{epochs}: Loss = {epoch_loss / len(train_loader):.4f}')
    return model

In [ ]:
print('Training CVAE...')
vae = CVAE(input_dim=4, latent_dim=2).to(device)
vae = train_vae(vae, train_loader, epochs=100)

In [ ]:
print('\nGenerating synthetic data with CVAE...')
n_oversample = 450
tiled_labels = np.tile(y_real, (n_oversample // len(y_real)) + 1)[:n_oversample]
synth_labels = torch.LongTensor(tiled_labels).to(device)

X_synthetic_vae_scaled = vae.generate(n_samples=n_oversample, labels=synth_labels)

# Apply Privacy Guard (threshold = 0.3)
safe_idx_vae = sanitize_synthetic_data(X_synthetic_vae_scaled, X_real_scaled, threshold=0.3)
X_synthetic_vae_scaled = X_synthetic_vae_scaled[safe_idx_vae]
y_synthetic_vae = tiled_labels[safe_idx_vae]

X_synthetic_vae = scaler.inverse_transform(X_synthetic_vae_scaled)

print(f'Retained {len(X_synthetic_vae)} samples after privacy filtering.')

## **6. Conditional Generative Adversarial Network (CGAN)** <a name="cgan"></a>

Next, we implement a tabular Conditional GAN consisting of:
- **Generator:** Takes a 10-dimensional Gaussian noise vector $\mathbf{z}$ concatenated with a one-hot class label $y$ and synthesizes 4 continuous features.
- **Discriminator:** Takes 4 continuous features concatenated with one-hot label $y$ and outputs real vs. synthetic classification probability.


In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim=10, output_dim=4, n_classes=3):
        super(Generator, self).__init__()
        self.label_emb = nn.Embedding(n_classes, n_classes)
        self.model = nn.Sequential(
            nn.Linear(latent_dim + n_classes, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, output_dim)
        )
    def forward(self, z, labels):
        c = self.label_emb(labels)
        return self.model(torch.cat([z, c], dim=1))

class Discriminator(nn.Module):
    def __init__(self, input_dim=4, n_classes=3):
        super(Discriminator, self).__init__()
        self.label_emb = nn.Embedding(n_classes, n_classes)
        self.model = nn.Sequential(
            nn.Linear(input_dim + n_classes, 32),
            nn.LeakyReLU(0.2),
            nn.Linear(32, 16),
            nn.LeakyReLU(0.2),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )
    def forward(self, x, labels):
        c = self.label_emb(labels)
        return self.model(torch.cat([x, c], dim=1))

In [ ]:
def train_gan(generator, discriminator, train_loader, epochs=400):
    g_optimizer = optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999))
    d_optimizer = optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))
    criterion = nn.BCELoss()
    
    for epoch in range(epochs):
        for x, y in train_loader:
            batch_size = x.size(0)
            x, y = x.to(device), y.to(device)
            
            # Train Discriminator
            d_optimizer.zero_grad()
            real_labels = torch.ones(batch_size, 1).to(device)
            fake_labels = torch.zeros(batch_size, 1).to(device)
            
            outputs = discriminator(x, y)
            d_loss_real = criterion(outputs, real_labels)
            
            z = torch.randn(batch_size, 10).to(device)
            fake_x = generator(z, y)
            outputs = discriminator(fake_x.detach(), y)
            d_loss_fake = criterion(outputs, fake_labels)
            
            d_loss = d_loss_real + d_loss_fake
            d_loss.backward()
            d_optimizer.step()
            
            # Train Generator
            g_optimizer.zero_grad()
            outputs = discriminator(fake_x, y)
            g_loss = criterion(outputs, real_labels)
            g_loss.backward()
            g_optimizer.step()
            
        if (epoch + 1) % 100 == 0:
            print(f'Epoch {epoch+1}/{epochs}: D_loss={d_loss.item():.4f}, G_loss={g_loss.item():.4f}')

In [ ]:
print('Training CGAN...')
generator = Generator().to(device)
discriminator = Discriminator().to(device)
train_gan(generator, discriminator, train_loader, epochs=400)

In [ ]:
print('\nGenerating synthetic data with CGAN...')
with torch.no_grad():
    z = torch.randn(n_oversample, 10).to(device)
    X_synthetic_gan_scaled = generator(z, synth_labels).cpu().numpy()

# Apply Privacy Guard (threshold = 0.3)
safe_idx_gan = sanitize_synthetic_data(X_synthetic_gan_scaled, X_real_scaled, threshold=0.3)
X_synthetic_gan_scaled = X_synthetic_gan_scaled[safe_idx_gan]
y_synthetic_gan = tiled_labels[safe_idx_gan]

X_synthetic_gan = scaler.inverse_transform(X_synthetic_gan_scaled)

print(f'Retained {len(X_synthetic_gan)} samples after privacy filtering.')


## **7. Fidelity & Utility Evaluation (TSTR Paradigm)** <a name="evaluation"></a>

How do we validate whether synthetic data is genuinely useful? We assess three core criteria:

1. **Mean Feature Error:** $\frac{1}{d} \sum_{j=1}^d |\bar{x}_{real, j} - \bar{x}_{synth, j}|$ (First-moment preservation).
2. **Correlation Matrix Error:** Mean absolute difference between empirical Pearson correlation matrices $\|\mathbf{R}_{real} - \mathbf{R}_{synth}\|_1$.
3. **Train on Synthetic, Test on Real (TSTR):**
   - We train a machine learning model ($k$-Nearest Neighbors or Logistic Regression) strictly using the **synthetic** records $\mathcal{D}_{syn}$.
   - We evaluate its predictive accuracy on the **real, unseen** test set $\mathcal{D}_{real}$.
   - High TSTR accuracy proves that the synthetic generator captured the true decision boundaries.


In [ ]:
def compute_quality_metrics(X_real, X_synthetic, y_real, y_synthetic, method_name):
    # 1. Statistical similarity (Mean Error)
    mean_real = X_real.mean(axis=0)
    mean_synth = X_synthetic.mean(axis=0)
    mean_error = np.mean(np.abs(mean_real - mean_synth))
    
    # 2. Correlation similarity
    corr_real = np.corrcoef(X_real.T)
    corr_synth = np.corrcoef(X_synthetic.T)
    corr_error = np.mean(np.abs(corr_real - corr_synth))
    
    # 3. Downstream utility (K-NN TSTR)
    knn_synth = KNeighborsClassifier(n_neighbors=3)
    knn_synth.fit(X_synthetic, y_synthetic)
    utility = knn_synth.score(X_real, y_real)
    
    return {
        'Method': method_name,
        'Mean Error': mean_error,
        'Corr Error': corr_error,
        'Utility (TSTR)': utility
    }


In [ ]:
print('\nComputing quality metrics...')
metrics_vae = compute_quality_metrics(X_real, X_synthetic_vae, y_real, y_synthetic_vae, 'CVAE')
metrics_gan = compute_quality_metrics(X_real, X_synthetic_gan, y_real, y_synthetic_gan, 'CGAN')

df_metrics = pd.DataFrame([metrics_vae, metrics_gan])
print(df_metrics)

## **8. Membership Inference Attack (MIA) Resistance** <a name="mia-defense"></a>

A privacy auditor assesses whether an adversary can determine if record $\mathbf{x}^*$ was in the training set by analyzing distances to synthetic samples.

We compute the **Nearest Neighbor Distance (NND)**:
- $\text{NND}_{syn \to real}$: Distance from each synthetic record to its closest real training neighbor.
- $\text{NND}_{real \to real}$: Distance from each real training record to its nearest real neighbor (baseline density).

If $\text{mean}(\text{NND}_{syn \to real}) \ge \text{mean}(\text{NND}_{real \to real})$, the synthetic dataset does not suffer from memorization collapse and successfully defends against membership inference.


In [ ]:
def nearest_neighbor_distance(data_point, dataset):
    distances = np.linalg.norm(dataset - data_point, axis=1)
    return distances.min()

def check_mia_resistance(X_synthetic, X_real, label=""):
    synth_to_real = [nearest_neighbor_distance(s, X_real) for s in X_synthetic]
    
    real_to_real = []
    for i, r in enumerate(X_real):
        dists = np.linalg.norm(X_real - r, axis=1)
        dists[i] = np.inf
        real_to_real.append(dists.min())
        
    s_mean, s_std = np.mean(synth_to_real), np.std(synth_to_real)
    r_mean, r_std = np.mean(real_to_real), np.std(real_to_real)
    
    print(f'\nMembership Inference Analysis ({label}):')
    print(f'  Synthetic -> Nearest Real: {s_mean:.4f} ± {s_std:.4f}')
    print(f'  Real -> Nearest Real: {r_mean:.4f} ± {r_std:.4f}')
    
    if s_mean > r_mean:
        print("  ✓ Synthetic samples are DISTINCT")
        print("  ✓ Hardened against membership inference")
    else:
        print("  ✗ Synthetic samples are SIMILAR (Privacy Risk)")


In [ ]:
check_mia_resistance(X_synthetic_vae, X_real, 'CVAE')
check_mia_resistance(X_synthetic_gan, X_real, 'CGAN')

## **9. Dimensionality Reduction & Distribution Verification (PCA)** <a name="pca"></a>

We project real and synthetic samples into the first two Principal Component dimensions to visually compare cluster coverage, class separation, and boundary overlap.


In [ ]:
pca = PCA(n_components=2)
X_real_pca = pca.fit_transform(X_real_scaled)
X_vae_pca = pca.transform(scaler.transform(X_synthetic_vae))
X_gan_pca = pca.transform(scaler.transform(X_synthetic_gan))

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.scatter(X_real_pca[:, 0], X_real_pca[:, 1], c=y_real, alpha=0.5, cmap='viridis')
plt.title('Real Data (PCA)')

plt.subplot(1, 3, 2)
plt.scatter(X_vae_pca[:, 0], X_vae_pca[:, 1], c=y_synthetic_vae, alpha=0.5, cmap='viridis')
plt.title('CVAE Synthetic (PCA)')

plt.subplot(1, 3, 3)
plt.scatter(X_gan_pca[:, 0], X_gan_pca[:, 1], c=y_synthetic_gan, alpha=0.5, cmap='viridis')
plt.title('CGAN Synthetic (PCA)')

plt.tight_layout()
plt.show()


## **10. Hands-on Exploratory Exercises** <a name="exercises"></a>

### **Exercise 1: Latent Space Dimensionality in CVAE**
Vary the latent dimension of the CVAE ($d_z \in \{1, 2, 8, 16\}$). How does latent compression affect:
1. Feature correlation error?
2. Downstream TSTR classification accuracy?
3. Memorization risk (nearest neighbor distance to real records)?

### **Exercise 2: Mode Collapse Analysis in Tabular GANs**
Remove the label conditioning from the GAN (unconditional GAN) and train it on Iris. Does the unconditional generator suffer from mode collapse (e.g. failing to generate the rarest or most compact class)? How does conditioning stabilize tabular generative modeling?

### **Exercise 3: Strictness of Privacy Guard Radius**
Evaluate downstream utility (TSTR) as you vary the privacy guard threshold $r_{min} \in \{0.1, 0.3, 0.5, 0.8\}$:
- At what threshold does privacy filtering begin to degrade downstream utility?
- Plot the Privacy-Utility frontier (Mean NND vs. TSTR Accuracy).

### **Exercise 4: Differentially Private Tabular Synthesis (DP-CGAN)**
Propose how you would integrate Differential Privacy into the CGAN training pipeline. Which component requires gradient clipping and noise addition: the Generator or the Discriminator? Why does the post-processing property of Differential Privacy guarantee that generator outputs remain $(\epsilon, \delta)$-DP?


## **11. Conclusion & Next Steps** <a name="conclusion"></a>

### **Core Takeaways**
1. **Synthetic Data Mitigates Privacy Vulnerabilities:** Artificially synthesizing tabular records from learned distributions avoids direct disclosure of sensitive real microdata.
2. **The TSTR Validation Paradigm:** Evaluating synthetic data using Train-on-Synthetic, Test-on-Real (TSTR) benchmarks provides rigorous empirical proof that synthetic records preserve actionable decision boundaries.
3. **Distance-Based Privacy Filtering:** Explicit post-generation distance rejection guards provide a practical, empirical layer of defense against memorization and Membership Inference Attacks (MIA).
4. **Bridging to Provable Privacy (Module 7):** While empirical distance filtering eliminates identical duplicates, formal privacy guarantees against arbitrary adversaries require training with **Differential Privacy (DP-SGD)**, which we will implement with Opacus in Module 7.

### **MITRE ATLAS Mapping**
- **Tactic:** [AML.TA0005: Exfiltration](https://atlas.mitre.org/tactics/AML.TA0005)
- **Technique:** [AML.T0024: Infer Training Data Membership](https://atlas.mitre.org/techniques/AML.T0024)
- **Mitigation:** [AML.M1004: Data Sanitization & Synthetic Data Generation](https://atlas.mitre.org/mitigations/AML.M1004)

### **Key References**
- Xu, L., Skoularidou, M., Cuesta-Infante, A., & Veeramachaneni, K. (2019). *Modeling Tabular Data using Conditional GAN (CTGAN)*. NeurIPS. [arXiv:1907.00503](https://arxiv.org/abs/1907.00503).
- Kingma, D. P., & Welling, M. (2013). *Auto-Encoding Variational Bayes*. ICLR. [arXiv:1312.6114](https://arxiv.org/abs/1312.6114).
- Jordon, J., Yoon, J., & van der Schaar, M. (2018). *PATE-GAN: Generating Synthetically Plausible Data with Differential Privacy Guarantees*. ICLR.
